# 03 — Join market odds, train, and backtest

Zero API calls.

Input:
- `data/processed/training_features.csv`
- `data/processed/historical_odds_aggregated.csv`

Output:
- market-only metrics;
- football-only metrics;
- football+market metrics;
- value backtest base;
- `data/processed/03_join_train_backtest_report_bundle.zip`


In [ ]:
# Cell 1. Imports and helpers.


from pathlib import Path
import json
import re
import zipfile
import time

import numpy as np
import pandas as pd

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP_UTC = pd.Timestamp.utcnow().isoformat()
SEED = 42

TEAM_REPLACEMENTS = {
    "USA": "United States",
    "US": "United States",
    "Korea Republic": "South Korea",
    "Türkiye": "Turkey",
    "Czechia": "Czech Republic",
    "Bosnia and Herzegovina": "Bosnia & Herzegovina",
    "Ivory Coast": "Côte d'Ivoire",
    "Cote d Ivoire": "Côte d'Ivoire",
}

def norm_team(name):
    if pd.isna(name):
        return ""
    name = str(name).strip()
    name = re.sub(r"\s+", " ", name)
    return TEAM_REPLACEMENTS.get(name, name)

def save_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            payload,
            f,
            indent=2,
            ensure_ascii=False,
            default=str,
        )
    return path

def infer_outcome(home_goals, away_goals):
    if pd.isna(home_goals) or pd.isna(away_goals):
        return np.nan
    if home_goals > away_goals:
        return 2
    if home_goals < away_goals:
        return 0
    return 1

def expected_score(rating_a, rating_b):
    return 1.0 / (1.0 + 10 ** (-(rating_a - rating_b) / 400.0))


from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

JOIN_DIR = PROCESSED_DIR / "join_train_backtest"
JOIN_DIR.mkdir(parents=True, exist_ok=True)

def multiclass_brier(y_true, proba):
    y_true = np.asarray(y_true).astype(int)
    one_hot = np.zeros_like(proba, dtype=float)
    one_hot[np.arange(len(y_true)), y_true] = 1.0
    return float(np.mean(np.sum((proba - one_hot) ** 2, axis=1)))

def evaluate_probs(name, y_true, proba):
    y_true = np.asarray(y_true).astype(int)
    pred = np.argmax(proba, axis=1)
    return {
        "model": name,
        "n": len(y_true),
        "accuracy": float(accuracy_score(y_true, pred)),
        "log_loss": float(log_loss(y_true, proba, labels=[0, 1, 2])),
        "brier": multiclass_brier(y_true, proba),
    }

def make_logit_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=3000,
            C=0.5,
            random_state=SEED,
        )),
    ])

print("Setup OK.")


In [ ]:
# Cell 2. Load files.

training_path = PROCESSED_DIR / "training_features.csv"
odds_path = PROCESSED_DIR / "historical_odds_aggregated.csv"

if not training_path.exists():
    raise FileNotFoundError("Run notebook 01 first.")

if not odds_path.exists():
    raise FileNotFoundError("Run notebook 02 first.")

training_features = pd.read_csv(training_path)
historical_odds = pd.read_csv(odds_path)

print("training_features:", training_features.shape)
print("historical_odds:", historical_odds.shape)

display(training_features.head())
display(historical_odds.head())


In [ ]:
# Cell 3. Prepare wide market features.

odds = historical_odds.copy()
odds["commence_time"] = pd.to_datetime(
    odds["commence_time"],
    utc=True,
    errors="coerce",
)
odds["date"] = odds["commence_time"].dt.date.astype(str)
odds["home_team"] = odds["home_team"].map(norm_team)
odds["away_team"] = odds["away_team"].map(norm_team)

base_cols = [
    "event_id",
    "date",
    "commence_time",
    "home_team",
    "away_team",
]

feature_cols = [
    "n_bookmakers",
    "avg_home_odds",
    "avg_draw_odds",
    "avg_away_odds",
    "best_home_odds",
    "best_draw_odds",
    "best_away_odds",
    "market_margin_avg",
    "market_p_away_devig",
    "market_p_draw_devig",
    "market_p_home_devig",
]

parts = []

for snap, group in odds.groupby("snapshot_label"):
    g = group[base_cols + feature_cols].copy()
    g = g.rename(columns={col: f"{snap}_{col}" for col in feature_cols})
    parts.append(g)

market_wide = None

for part in parts:
    if market_wide is None:
        market_wide = part
    else:
        market_wide = market_wide.merge(
            part,
            on=base_cols,
            how="outer",
        )

if market_wide is None:
    market_wide = pd.DataFrame()

for outcome in ["home", "draw", "away"]:
    p24 = f"T_minus_24h_market_p_{outcome}_devig"
    p1 = f"T_minus_1h_market_p_{outcome}_devig"
    if p24 in market_wide.columns and p1 in market_wide.columns:
        market_wide[f"move_p_{outcome}_24h_to_1h"] = (
            market_wide[p1] - market_wide[p24]
        )

market_wide.to_csv(
    JOIN_DIR / "market_features_wide.csv",
    index=False,
)

print("market_wide:", market_wide.shape)
display(market_wide.head())


In [ ]:
# Cell 4. Join.

train = training_features.copy()
train["date"] = pd.to_datetime(
    train["date"],
    errors="coerce",
).dt.date.astype(str)
train["home_team"] = train["home_team"].map(norm_team)
train["away_team"] = train["away_team"].map(norm_team)

joined = train.merge(
    market_wide,
    on=["date", "home_team", "away_team"],
    how="inner",
    suffixes=("", "_market"),
)

joined.to_csv(
    JOIN_DIR / "training_features_joined_market.csv",
    index=False,
)

print("joined:", joined.shape)
display(joined.head())

if len(joined) == 0:
    print("WARNING: empty join. Check date/team naming.")


In [ ]:
# Cell 5. Market-only metrics.

metrics_rows = []

if len(joined) > 0:
    joined["outcome"] = joined["outcome"].astype(int)

    for snap in ["T_minus_24h", "T_minus_3h", "T_minus_1h"]:
        cols = [
            f"{snap}_market_p_away_devig",
            f"{snap}_market_p_draw_devig",
            f"{snap}_market_p_home_devig",
        ]
        if all(c in joined.columns for c in cols):
            tmp = joined.dropna(subset=cols + ["outcome"]).copy()
            if len(tmp) > 0:
                metrics_rows.append(
                    evaluate_probs(
                        f"market_only_{snap}",
                        tmp["outcome"].to_numpy(),
                        tmp[cols].to_numpy(),
                    )
                )

market_baseline_metrics = pd.DataFrame(metrics_rows)

market_baseline_metrics.to_csv(
    JOIN_DIR / "market_baseline_metrics.csv",
    index=False,
)

display(market_baseline_metrics)


In [ ]:
# Cell 6. Football-only and football+market models.

model_rows = []

if len(joined) < 30:
    print("Not enough joined rows for stable training:", len(joined))
else:
    data = joined.copy()
    data["outcome"] = data["outcome"].astype(int)
    data["date_dt"] = pd.to_datetime(data["date"], errors="coerce")

    split_date = data["date_dt"].quantile(0.75)

    train_df = data[data["date_dt"] <= split_date].copy()
    test_df = data[data["date_dt"] > split_date].copy()

    exclude = {
        "event_id",
        "date",
        "date_dt",
        "commence_time",
        "home_team",
        "away_team",
        "tournament",
        "outcome",
        "home_score",
        "away_score",
    }

    numeric_cols = [
        col for col in data.columns
        if col not in exclude
        and pd.api.types.is_numeric_dtype(data[col])
    ]

    market_cols = [
        col for col in numeric_cols
        if "market_" in col
        or "_odds" in col
        or "n_bookmakers" in col
        or "move_" in col
    ]

    football_cols = [
        col for col in numeric_cols
        if col not in market_cols
    ]

    feature_sets = {
        "football_only_logit": football_cols,
        "football_plus_market_logit": football_cols + market_cols,
    }

    for name, features in feature_sets.items():
        if not features:
            continue

        model = make_logit_pipeline()
        model.fit(train_df[features], train_df["outcome"])

        raw = model.predict_proba(test_df[features])
        proba = np.zeros((len(test_df), 3))

        for i, cls in enumerate(model.named_steps["model"].classes_):
            proba[:, int(cls)] = raw[:, i]

        row = evaluate_probs(
            name,
            test_df["outcome"].to_numpy(),
            proba,
        )
        row["n_train"] = len(train_df)
        row["n_test"] = len(test_df)
        row["n_features"] = len(features)
        model_rows.append(row)

    pd.Series(football_cols).to_csv(
        JOIN_DIR / "football_feature_columns.csv",
        index=False,
    )
    pd.Series(market_cols).to_csv(
        JOIN_DIR / "market_feature_columns.csv",
        index=False,
    )

football_vs_market_model_metrics = pd.DataFrame(model_rows)

football_vs_market_model_metrics.to_csv(
    JOIN_DIR / "football_vs_market_model_metrics.csv",
    index=False,
)

display(football_vs_market_model_metrics)


In [ ]:
# Cell 7. Historical value backtest base.

value_rows = []

if len(joined) > 0:
    for _, r in joined.iterrows():
        base = {
            "date": r["date"],
            "home_team": r["home_team"],
            "away_team": r["away_team"],
            "outcome": int(r["outcome"]),
        }

        for outcome_name, class_id in [
            ("away", 0),
            ("draw", 1),
            ("home", 2),
        ]:
            p24 = r.get(f"T_minus_24h_market_p_{outcome_name}_devig", np.nan)
            p1 = r.get(f"T_minus_1h_market_p_{outcome_name}_devig", np.nan)
            o24 = r.get(f"T_minus_24h_best_{outcome_name}_odds", np.nan)
            o1 = r.get(f"T_minus_1h_best_{outcome_name}_odds", np.nan)

            value_rows.append({
                **base,
                "selection": outcome_name,
                "selection_class": class_id,
                "market_p_24h": p24,
                "market_p_1h": p1,
                "best_odds_24h": o24,
                "best_odds_1h": o1,
                "selection_won": int(int(r["outcome"]) == class_id),
                "clv_proxy_prob": (
                    p1 - p24
                    if pd.notna(p1) and pd.notna(p24)
                    else np.nan
                ),
                "clv_proxy_odds": (
                    o24 / o1 - 1.0
                    if pd.notna(o24)
                    and pd.notna(o1)
                    and o1 > 0
                    else np.nan
                ),
            })

historical_value_backtest_base = pd.DataFrame(value_rows)

historical_value_backtest_base.to_csv(
    JOIN_DIR / "historical_value_backtest_base.csv",
    index=False,
)

display(historical_value_backtest_base.head(30))
print("value rows:", len(historical_value_backtest_base))


In [ ]:
# Cell 8. Save join/train/backtest report.

summary = {
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
    "training_rows": int(len(training_features)),
    "historical_odds_rows": int(len(historical_odds)),
    "market_wide_rows": int(len(market_wide)),
    "joined_rows": int(len(joined)),
    "market_metrics_rows": int(len(market_baseline_metrics)),
    "model_metrics_rows": int(len(football_vs_market_model_metrics)),
    "value_rows": int(len(historical_value_backtest_base)),
}

save_json(JOIN_DIR / "summary.json", summary)

zip_path = PROCESSED_DIR / "03_join_train_backtest_report_bundle.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file in JOIN_DIR.rglob("*"):
        if file.is_file():
            zf.write(file, arcname=file.relative_to(JOIN_DIR))

print("Created:", zip_path.resolve())
